# Two view corner averaging

**CPU runtime is sufficient. Run all four code cells.**

This notebook uses the returned original and horizontally reflected predictions.
It keeps every original detection box, score and record identifier. A same-class,
mutual-best predicted-box match with overlap at least 0.50 supplies an alternative
corner pair. If both original and alternative pairs are valid at corner score 0.50,
their corresponding coordinates are averaged equally. Other records stay unchanged.

No annotation selects a view, match, weight or cutoff. The annotations are used
afterward to measure performance. Original scores define a fixed comparison
cohort; they are not calibrated confidence in the averaged coordinates.
This is an exploratory analysis of the existing 20 development photographs.
It performs no model inference or training.


## 1 Load the published experiment source

The notebook uses one immutable source revision. Choose a fresh
`BAOMAI_CORNER_AVERAGE_BASE` for a separate run; incompatible existing files
are preserved. Python and its standard library are sufficient.


In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, urllib.request

SOURCE_COMMIT = "5b1650b8fa3959a0917680305d7b499af783c5aa"
DRIVER_SHA256 = "c29b1868a74128756feff39b78975ea8397c4f1b88e576b38461b756e5e86bfc"
BASE = Path(os.environ.get("BAOMAI_CORNER_AVERAGE_BASE", "/content/baomai-corner-average")).resolve()
SOURCE, WORK, CACHE = BASE/"source", BASE/"results", BASE/"cache"
BASE.mkdir(parents=True, exist_ok=True)
ENV = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1", "GIT_TERMINAL_PROMPT": "0"}
def run(command, cwd=None):
    subprocess.run(list(map(str, command)), cwd=cwd, env=ENV, check=True)
def git(*args):
    return subprocess.check_output(["git", *args], cwd=SOURCE, env=ENV, text=True).strip()
if not (SOURCE/".git").exists():
    if SOURCE.exists() and any(SOURCE.iterdir()):
        raise RuntimeError("Nonempty source preserved. Choose a fresh base folder.")
    run(["git", "init", SOURCE])
    run(["git", "remote", "add", "origin", "https://github.com/Siuuuers/dl.git"], SOURCE)
    run(["git", "fetch", "--depth=1", "--filter=blob:none", "origin", SOURCE_COMMIT], SOURCE)
    run(["git", "sparse-checkout", "init", "--cone"], SOURCE)
    run(["git", "sparse-checkout", "set", "scripts", "data"], SOURCE)
    run(["git", "checkout", "--detach", SOURCE_COMMIT], SOURCE)
if git("rev-parse", "HEAD") != SOURCE_COMMIT or git("status", "--porcelain", "--untracked-files=all"):
    raise RuntimeError("Changed source preserved. Choose a fresh base folder.")
DRIVER = SOURCE/"scripts/run_corner_average_r58.py"
assert hashlib.sha256(DRIVER.read_bytes()).hexdigest() == DRIVER_SHA256
print("Source verified:", SOURCE_COMMIT)


## 2 Fetch and verify the returned predictions

The exact reflection result archive is downloaded from its immutable results
commit. Its size and SHA-256 are checked before analysis. The driver additionally
verifies the member inventory, source identity, geometry and reference labels.


In [ ]:
CACHE.mkdir(parents=True, exist_ok=True)
RETURN_SHA256 = "88689ce9f3d03178683ea98ce98d50f5ddc4e07ba1fb24262e7820e9488bd114"
RETURN_BYTES = 2931102
URL = "https://raw.githubusercontent.com/Siuuuers/dl/dfe5392109433fcee51f41716710001bb123a3d5/results/a100/baomai-flip-inference-r56-a100-001/baomai-flip-inference-r56-results.zip"
INPUT = CACHE/"reflection-results.zip"
if INPUT.exists():
    payload = INPUT.read_bytes()
else:
    with urllib.request.urlopen(URL, timeout=90) as response:
        payload = response.read(RETURN_BYTES+1)
if len(payload) != RETURN_BYTES or hashlib.sha256(payload).hexdigest() != RETURN_SHA256:
    raise RuntimeError("Result archive differs; existing cache was preserved.")
if not INPUT.exists():
    INPUT.write_bytes(payload)
def command(action):
    return [sys.executable, "-B", DRIVER, action, "--root", SOURCE,
            "--source-commit", SOURCE_COMMIT, "--script-sha256", DRIVER_SHA256,
            "--archive", INPUT, "--work", WORK]
print("Verified reflection archive:", len(payload), "bytes")


## 3 Compare coordinates with all detection records fixed

The original and reflected saved tables are reproduced before interpreting
the new coordinate average. All eligible two-corner targets remain in the
denominator, including missed fronts. Both five- and ten-pixel pair criteria
are reported, together with gained and lost targets and accepted-output groups.


In [ ]:
run(command("run"))
summary = json.loads((WORK/"summary.json").read_text())
print("Association counts:", summary["association_counts"])
print("reference | coordinates | misses | extras | duplicates | pair5 | pair10")
for reference in ("frozen666", "r50"):
    for arm in ("control", "reflection", "average"):
        item = summary["references"][reference]["arms"][arm]
        d, p = item["detection"], item["pairs"]
        print(reference, arm, d["fn"], d["fp"], d["duplicate_fp"],
              f"{p['within5']}/{p['eligible']}", f"{p['within10']}/{p['eligible']}")
    print("Averaging gains and losses:", summary["references"][reference]["paired_changes"])


## 4 Verify and download the complete result

Replay verifies the output bundle against a fresh calculation. The archive
includes coordinate predictions, view associations, all target changes,
complete summary tables, source identities and a member hash inventory.
A better five-pixel result can accompany a worse ten-pixel result; read both.


In [ ]:
run(command("verify"))
ARCHIVE = WORK/"baomai-corner-average-r58-results.zip"
from IPython.display import FileLink, display
try:
    from google.colab import files
except ImportError:
    display(FileLink(str(ARCHIVE)))
else:
    files.download(str(ARCHIVE))
